# Configuration


In [3]:
model_name="unsloth/Qwen3.5-0.8B"
max_seq_length=2048
load_in_4bit=False
COPIES = 1   

LANG="Banglish"

main_training_data_path="/kaggle/input/datasets/anowersha/jev-version1/en_3000match.csv"
intent_matching_data_path="/kaggle/input/datasets/anowersha/jev-version1/massive_intent_mapping.csv"
bangla_training_data_path="/kaggle/input/datasets/anowersha/jev-version1/bn_3000_clean_unique.csv"
cm_training_data_path="/kaggle/input/datasets/anowersha/jev-version1/train_set_codemixed_massive60_fixed.csv"
banglish_training_data_path="/kaggle/input/datasets/anowersha/jev-version1/banglish_massive_train.csv"

merge_needed=True


INSTR = "What is the user's main intent?"
num_train_epochs=2


EN_testing_dataset_with_intent_name="/kaggle/input/datasets/anowersha/jev-version1/test_en_2000.csv"
BN_testing_dataset_with_intent_name="/kaggle/input/datasets/anowersha/jev-version1/test_bn_2000.csv"
CM_testing_dataset_with_intent_name="/kaggle/input/datasets/anowersha/jev-version1/test_set_codemixed_massive60_fixed.csv"
Banglish_testing_dataset_with_intent_name="/kaggle/input/datasets/anowersha/jev-version1/banglish_massive_test.csv"
base_model_inference=True   # run the evaluation block in the 'Base model Inference' section



import os
os.environ["UNSLOTH_COMPILE_DISABLE"] = "1"
from kaggle_secrets import UserSecretsClient
hf_token = UserSecretsClient().get_secret("HF_TOKEN")   # add it under Add-ons -> Secrets
os.environ["HF_TOKEN"] = hf_token


WILL_FT=False


remote_repo="Kashmary/qwen3.5_combined"

# Evaluate a saved model instead of the base model.
# True: load the trained joint model from saved_model_path and skip get_peft_model.
LOAD_SAVED_MODEL=True
saved_model_path=remote_repo
EVAL_RUN_NAME="joint_saved"   # output folder becomes results_{EVAL_RUN_NAME}_{LANG}


### Installation

In [ ]:
# %%capture
import os, importlib.util
!pip install --upgrade -qqq uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try: import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
    except: _numpy = "numpy"; _pil = "pillow"
    !uv pip install -qqq \
        "torch==2.8.0" "triton>=3.3.0" {_numpy} {_pil} torchvision "torchaudio==2.8.0" bitsandbytes xformers==0.0.32.post2 \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth"
    !uv pip install -qqq --no-deps "torchcodec==0.7.0"
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth
!uv pip install --upgrade --no-deps "tokenizers>=0.22.0,<=0.23.0" trl==0.22.2 unsloth unsloth_zoo
!uv pip install transformers==5.2.0
!unsloth install-kernels
!uv pip install --no-deps --upgrade "torchao>=0.16.0"

# Packages import

In [ ]:
from unsloth import FastDecisionModel, DecisionTrainer, is_bfloat16_supported
import torch
import os, json, time
import numpy as np, pandas as pd
from collections import Counter
from tqdm.auto import tqdm
from sklearn.metrics import (precision_recall_fscore_support, classification_report,
                             cohen_kappa_score, matthews_corrcoef)

import pandas as pd
import json, random
from datasets import Dataset
from transformers import TrainingArguments
import zipfile
from pathlib import Path

# Model Loading

In [ ]:
model, tokenizer = FastDecisionModel.from_pretrained(
    model_name = saved_model_path if LOAD_SAVED_MODEL else model_name,
    max_seq_length = max_seq_length, # Longest input. Long inputs keep the question and options.
    load_in_4bit = load_in_4bit, # 4 bit quantization to reduce memory
)

In [ ]:
if not LOAD_SAVED_MODEL:   # a saved model already has its LoRA adapter and head
    model = FastDecisionModel.get_peft_model(
        model,
        r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
        lora_alpha = 16,
        lora_dropout = 0, # Supports any, but = 0 is optimized
        use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
        random_state = 3407,
    )

# Base model Inference

In [6]:
import pandas as pd

cols = ['id', 'utt', 'intent']

def map_intent(df, intent_map):
    df = df.merge(intent_map, left_on='intent', right_on='intent_id', how='left')
    df = df.drop(columns=['intent_id', 'intent'])
    return df.rename(columns={'intent_label': 'intent'})

if merge_needed:
    intent_map = pd.read_csv(intent_matching_data_path)
    df1 = map_intent(pd.read_csv(main_training_data_path), intent_map)
    df3 = map_intent(pd.read_csv(bangla_training_data_path), intent_map)
else:
    df1 = pd.read_csv(main_training_data_path)
    df3 = None

df5 = pd.read_csv(cm_training_data_path)
df6 = pd.read_csv(banglish_training_data_path)

# Combine, keeping only the three columns
df6 = df6.rename(columns={'Id': 'id'})
frames = [df1, df3, df5, df6]
combined_df = pd.concat(
    [f[cols] for f in frames if f is not None],
    ignore_index=True
)

df1= df1 = combined_df

In [ ]:
DESC = {
 "alarm_query": "Ask about existing alarms, including which alarms are set or when the next one rings.",
 "alarm_remove": "Cancel or delete an existing alarm.",
 "alarm_set": "Set or create a new alarm.",
 "audio_volume_down": "Lower the audio volume.",
 "audio_volume_mute": "Mute or silence the audio.",
 "audio_volume_other": "Other volume requests that are not a plain increase, decrease, or mute, such as setting a specific volume level or setting the volume to maximum.",
 "audio_volume_up": "Increase the audio volume.",
 "calendar_query": "Ask about existing calendar events, appointments, or meetings. Not for alarms or the current time.",
 "calendar_remove": "Delete or cancel a calendar event or appointment.",
 "calendar_set": "Create or schedule a calendar event, appointment, or meeting.",
 "cooking_query": "Ask cooking-related questions about ingredients, cooking time, or substitutions, without requesting a complete recipe.",
 "cooking_recipe": "Ask for a recipe or step-by-step instructions on how to prepare a dish.",
 "datetime_convert": "Convert a time between time zones or geographical locations.",
 "datetime_query": "Ask for the current date, day, or time, including the time in a particular location. Not for calendar events.",
 "email_addcontact": "Add a person to the user's contacts.",
 "email_query": "Check, retrieve, or read emails, including new emails or emails from a particular sender. Not for looking up a contact's address.",
 "email_querycontact": "Ask for a contact's email address or other contact details.",
 "email_sendemail": "Compose or send an email.",
 "general_greet": "Greet the assistant or initiate casual conversation, such as saying hello or asking how it is.",
 "general_joke": "Ask the assistant to tell a joke.",
 "general_quirky": "Casual conversation, requests for opinions, or unusual or unclear requests. Use only when no other intent fits.",
 "iot_cleaning": "Start or control a cleaning device, such as a robot vacuum cleaner.",
 "iot_coffee": "Make coffee or control a coffee machine.",
 "iot_hue_lightchange": "Change the colour of smart lights.",
 "iot_hue_lightdim": "Dim smart lights or reduce their brightness.",
 "iot_hue_lightoff": "Turn smart lights off.",
 "iot_hue_lighton": "Turn smart lights on.",
 "iot_hue_lightup": "Increase the brightness of smart lights.",
 "iot_wemo_off": "Turn off a smart plug or a device controlled by a smart switch. Not for lights.",
 "iot_wemo_on": "Turn on a smart plug or a device controlled by a smart switch. Not for lights.",
 "lists_createoradd": "Create a list or add an item to an existing list, such as a shopping list or to-do list.",
 "lists_query": "Ask to view or retrieve the items on an existing list.",
 "lists_remove": "Remove an item from a list or delete an entire list.",
 "music_dislikeness": "Express dislike for a song or music artist.",
 "music_likeness": "Express liking or appreciation for a song or music artist.",
 "music_query": "Ask which song or artist is playing, or ask for information about a song or artist. Not a request to play music.",
 "music_settings": "Change music playback settings, such as repeat or shuffle.",
 "news_query": "Ask for news, current headlines, or news updates.",
 "play_audiobook": "Request playback of an audiobook.",
 "play_game": "Ask to play a game with the assistant.",
 "play_music": "Request playback of a song, artist, album, or playlist. Choose among the play labels by the type of media named.",
 "play_podcasts": "Request playback of a podcast.",
 "play_radio": "Request playback of a radio station.",
 "qa_currency": "Ask about currency exchange rates or currency conversion.",
 "qa_definition": "Ask for the meaning or definition of a word or term.",
 "qa_factoid": "Ask a factual question about people, places, or general world knowledge.",
 "qa_maths": "Ask for a mathematical calculation or a unit conversion. Currency conversion is qa_currency.",
 "qa_stock": "Ask about stock prices or share-related information.",
 "recommendation_events": "Ask for recommendations of events or activities to attend.",
 "recommendation_locations": "Ask for recommendations of places to visit or eat at, including restaurants or nearby locations. Not for placing a food order.",
 "recommendation_movies": "Ask for movie or television show recommendations or showtimes.",
 "social_post": "Create or publish a post on social media.",
 "social_query": "Check social media feeds, posts, or updates.",
 "takeaway_order": "Place a new food order for delivery or takeaway.",
 "takeaway_query": "Ask about an existing food order or obtain information about a restaurant, such as order status or opening hours.",
 "transport_query": "Ask about routes, directions, or travel duration. Not for traffic conditions.",
 "transport_taxi": "Request a taxi or ride-hailing service.",
 "transport_ticket": "Book or purchase a transport ticket for a train, bus, flight, or other transport service.",
 "transport_traffic": "Ask about traffic conditions.",
 "weather_query": "Ask about current weather conditions or a weather forecast.",
}

In [ ]:


INSTR = "What is the user's main intent?"      # must match training exactly

# DESC = {...}   # <- re-run the cell that defines the 60-label DESC dict (or paste it here)
ALL = sorted(DESC)
assert len(ALL) == 60

In [ ]:
if base_model_inference:
    DATASETS = {                                   # name -> csv path (needs columns: utt, intent)
        "EN": EN_testing_dataset_with_intent_name,
        "BN": BN_testing_dataset_with_intent_name,
        "CM": CM_testing_dataset_with_intent_name,
        "Banglish": Banglish_testing_dataset_with_intent_name,
        
    }
    N_SAMPLES = {"EN": 2000, "BN": 2000, "CM": 2000, "Banglish":2000}   # None = use every row
    REMOVE_TRAIN_OVERLAP = True     # keep identical for base and fine-tuned runs so both score the same rows
    RUN_NAME = EVAL_RUN_NAME if LOAD_SAVED_MODEL else "base_model"
    OUT_DIR = f"/kaggle/working/results_{RUN_NAME}_{LANG}"
    os.makedirs(OUT_DIR, exist_ok=True)
    DOM = lambda l: l.split("_")[0]
    # Needs from earlier cells: model, tokenizer, ALL, DESC, INSTR, df1, base_model_inference
    # ==================================================
    
    FastDecisionModel.for_inference(model)
    
    # ---------- model footprint (same for every dataset) ----------
    TOTAL_PARAMS = sum(p.numel() for p in model.parameters())
    TRAINABLE_PARAMS = sum(p.numel() for p in model.parameters() if p.requires_grad)
    GPU_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"
    WARMUP = 5                      # untimed predictions before timing starts
    
    def predict_full(utt, labels=None):
        labels = labels or ALL
        questions = {"intent": {"type": "choice", "instructions": INSTR,
                                "criteria": {l: DESC[l] for l in labels}}}
        res = FastDecisionModel.predict(model, tokenizer, utt, questions)["intent"]
        answer = res.get("answer", res.get("label"))
        ranked = sorted(res["probabilities"].items(), key=lambda kv: -kv[1])
        return answer, ranked
    
    def ece_score(conf, correct, bins=10):
        edges = np.linspace(0, 1, bins + 1); e = 0.0
        for lo, hi in zip(edges[:-1], edges[1:]):
            m = (conf > lo) & (conf <= hi)
            if m.any(): e += m.mean() * abs(correct[m].mean() - conf[m].mean())
        return e
    
    def wilson(k, n, z=1.96):
        if n == 0: return 0.0, 0.0
        p = k / n; d = 1 + z * z / n
        c = (p + z * z / (2 * n)) / d
        h = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
        return c - h, c + h
    
    def load_partial(path):
        done = {}
        if os.path.exists(path):
            with open(path, encoding="utf-8") as f:
                for line in f:
                    try:
                        r = json.loads(line); done[(r["row_id"], r["utt"])] = r
                    except (json.JSONDecodeError, KeyError):
                        pass                                   # last line may be cut by a crash
        return done
    
    def run_dataset(name, path, n):
        df = pd.read_csv(path).dropna(subset=["utt", "intent"]).reset_index(drop=True)
        bad = set(df.intent) - set(ALL)
        assert not bad, f"{name}: labels without a definition: {bad}"
        if torch.cuda.is_available(): torch.cuda.reset_peak_memory_stats()
        removed = 0
        if REMOVE_TRAIN_OVERLAP:
            keep = ~df.utt.isin(set(df1.utt)); removed = int((~keep).sum()); df = df[keep].reset_index(drop=True)
        total_rows = len(df)
        sub = (df.sample(min(n, len(df)), random_state=0) if n else df).reset_index(drop=True)
    
        # ---------- predictions: appended line by line, resumable ----------
        ckpt = f"{OUT_DIR}/{name}_predictions_partial.jsonl"
        done = load_partial(ckpt)
        if done: print(f"{name}: resuming, {len(done)} predictions already saved")
        rows = list(done.values())
        correct_ct = sum(r["pred"] == r["intent"] for r in rows)
        todo = [(i, u, g) for i, (u, g) in enumerate(zip(sub.utt, sub.intent)) if (i, u) not in done]
    
        for utt in list(sub.utt[:WARMUP]) if todo else []:      # warm-up: GPU kernels and caches
            predict_full(utt)
        if torch.cuda.is_available(): torch.cuda.synchronize()
    
        with open(ckpt, "a", encoding="utf-8") as f:
            bar = tqdm(todo, total=len(todo), desc=name, dynamic_ncols=True)
            for i, utt, gold in bar:
                t1 = time.perf_counter()
                pred, ranked = predict_full(utt)
                if torch.cuda.is_available(): torch.cuda.synchronize()
                r = {"row_id": i, "utt": utt, "intent": gold, "pred": pred, "conf": ranked[0][1],
                     "top5": [l for l, _ in ranked[:5]], "sec": time.perf_counter() - t1}
                rows.append(r); correct_ct += int(pred == gold)
                bar.set_postfix(acc=f"{correct_ct / len(rows):.3f}")
                f.write(json.dumps(r, ensure_ascii=False) + "\n")
                f.flush(); os.fsync(f.fileno())
    
        res = pd.DataFrame(rows).sort_values("row_id").reset_index(drop=True)
        secs = res.sec.sum()
        lat_ms = res.sec.values.astype(float) * 1000         # per-utterance latency, batch size 1
        peak_mem = torch.cuda.max_memory_allocated() / 1024**3 if torch.cuda.is_available() else float("nan")
        y, p = res.intent.values, res.pred.values
        ok = (y == p); conf = res.conf.values.astype(float)
    
        # ---------- metrics ----------
        labs = sorted(set(y))
        P, R, F, _ = precision_recall_fscore_support(y, p, labels=labs, average=None, zero_division=0)
        macro = precision_recall_fscore_support(y, p, labels=labs, average="macro", zero_division=0)[:3]
        weighted = precision_recall_fscore_support(y, p, labels=labs, average="weighted", zero_division=0)[:3]
        acc = ok.mean(); lo, hi = wilson(int(ok.sum()), len(ok))
        top = {k: np.mean([g in t5[:k] for g, t5 in zip(y, res.top5)]) for k in (1, 3, 5)}
        domain_ok = np.array([DOM(a) == DOM(b) for a, b in zip(y, p)])
        invalid = int((~res.pred.isin(ALL)).sum())
        rng = np.random.default_rng(0); n_ = len(res); boot_acc, boot_f1 = [], []
        for _ in range(500):
            ix = rng.integers(0, n_, n_)
            boot_acc.append((y[ix] == p[ix]).mean())
            boot_f1.append(precision_recall_fscore_support(y[ix], p[ix], labels=labs, average="macro", zero_division=0)[2])
        err = res[~ok]
        intra = int(domain_ok[~ok].sum()); cross = int((~ok).sum()) - intra
        pairs = Counter(zip(err.intent, err.pred)).most_common(15)
        dom_acc = (res.assign(ok=ok, dom=[DOM(l) for l in y]).groupby("dom").ok
                   .agg(["mean", "count"]).sort_values("mean"))
        cls = pd.DataFrame({"label": labs, "precision": P, "recall": R, "f1": F,
                            "support": [int((y == l).sum()) for l in labs]}).sort_values("f1")
        cm = pd.crosstab(res.intent, res.pred)
    
        summary = {"run": RUN_NAME, "dataset": name, "n_eval": len(res), "accuracy": acc,
                   "acc_ci_low": lo, "acc_ci_high": hi, "top3_acc": top[3], "top5_acc": top[5],
                   "macro_P": macro[0], "macro_R": macro[1], "macro_F1": macro[2], "weighted_F1": weighted[2],
                   "kappa": cohen_kappa_score(y, p), "mcc": matthews_corrcoef(y, p),
                   "domain_acc": domain_ok.mean(), "ECE": ece_score(conf, ok.astype(float)),
                   "mean_conf_correct": conf[ok].mean() if ok.any() else np.nan,
                   "mean_conf_wrong": conf[~ok].mean() if (~ok).any() else np.nan,
                   "errors": int((~ok).sum()), "intra_domain_err": intra, "cross_domain_err": cross,
                   "invalid_preds": invalid, "sec_per_utt": secs / len(res),
                   "mean_latency_ms_per_sample": lat_ms.mean(), "median_latency_ms": float(np.median(lat_ms)),
                   "p95_latency_ms": float(np.percentile(lat_ms, 95)),
                   "throughput_samples_per_sec": 1000 / lat_ms.mean(),
                   "peak_gpu_mem_gb": peak_mem, "total_params_M": TOTAL_PARAMS / 1e6,
                   "trainable_params_M": TRAINABLE_PARAMS / 1e6, "gpu": GPU_NAME, "batch_size": 1}
    
        # ---------- text report ----------
        L = [f"RUN: {RUN_NAME} | DATASET: {name} | file: {path}",
             f"rows after filtering: {total_rows} (train-overlap removed: {removed}) | evaluated: {len(res)}",
             f"inference time: {secs/60:.1f} min ({secs/len(res):.2f} s/utt) | label list: all {len(ALL)} labels per query",
             f"latency (batch 1, {GPU_NAME}): mean {lat_ms.mean():.1f} ms | median {np.median(lat_ms):.1f} ms | p95 {np.percentile(lat_ms, 95):.1f} ms | throughput {1000 / lat_ms.mean():.2f} utt/s",
             f"peak GPU memory: {peak_mem:.2f} GB | parameters: {TOTAL_PARAMS / 1e6:.1f}M total, {TRAINABLE_PARAMS / 1e6:.2f}M trainable", "",
             "=== OVERALL ===",
             f"Accuracy (= micro-F1)     : {acc:.4f}  (95% Wilson CI {lo:.4f}-{hi:.4f}; bootstrap {np.percentile(boot_acc,2.5):.4f}-{np.percentile(boot_acc,97.5):.4f})",
             f"Top-3 / Top-5 accuracy    : {top[3]:.4f} / {top[5]:.4f}",
             f"Macro    P / R / F1       : {macro[0]:.4f} / {macro[1]:.4f} / {macro[2]:.4f}  (F1 bootstrap CI {np.percentile(boot_f1,2.5):.4f}-{np.percentile(boot_f1,97.5):.4f})",
             f"Weighted P / R / F1       : {weighted[0]:.4f} / {weighted[1]:.4f} / {weighted[2]:.4f}",
             f"Cohen's kappa / MCC       : {summary['kappa']:.4f} / {summary['mcc']:.4f}",
             f"Domain-level accuracy     : {summary['domain_acc']:.4f}",
             f"Invalid (non-label) preds : {invalid}", "",
             "=== ERROR ANALYSIS ===",
             f"Total errors: {summary['errors']} | same-domain: {intra} | cross-domain: {cross}",
             f"Calibration: ECE={summary['ECE']:.4f} | mean confidence correct={summary['mean_conf_correct']:.3f}, wrong={summary['mean_conf_wrong']:.3f}", "",
             "Top confused pairs (gold -> pred : count):"]
        L += [f"  {a} -> {b} : {c}" for (a, b), c in pairs]
        L += ["", "Accuracy per domain (worst first):", dom_acc.round(4).to_string(), "",
              "10 weakest labels by F1:", cls.head(10).round(4).to_string(index=False), "",
              "10 strongest labels by F1:", cls.tail(10).round(4).to_string(index=False), "",
              "Sample errors (first 25):", err[["utt", "intent", "pred", "conf"]].head(25).round(3).to_string(index=False), "",
              "=== FULL PER-CLASS REPORT ===", classification_report(y, p, labels=labs, digits=4, zero_division=0)]
        txt = "\n".join(L)
    
        open(f"{OUT_DIR}/{name}_report.txt", "w", encoding="utf-8").write(txt)
        cls.to_csv(f"{OUT_DIR}/{name}_per_class.csv", index=False)
        cm.to_csv(f"{OUT_DIR}/{name}_confusion_matrix.csv")
        res.drop(columns=["top5", "sec"]).to_csv(f"{OUT_DIR}/{name}_predictions.csv", index=False, encoding="utf-8-sig")
        json.dump(summary, open(f"{OUT_DIR}/{name}_summary.json", "w"), default=float)   # marks dataset as finished
        print(txt[:2500])
        return summary
    
    # ---------- run all datasets (skips finished ones, resumes the interrupted one) ----------
    all_sum = []
    for name, path in DATASETS.items():
        sj = f"{OUT_DIR}/{name}_summary.json"
        if os.path.exists(sj):
            print(f"{name} already finished, skipping"); all_sum.append(json.load(open(sj))); continue
        all_sum.append(run_dataset(name, path, N_SAMPLES.get(name)))
        pd.DataFrame(all_sum).to_csv(f"{OUT_DIR}/summary_all_datasets.csv", index=False)
    
    summary_df = pd.DataFrame(all_sum)
    summary_df.to_csv(f"{OUT_DIR}/summary_all_datasets.csv", index=False)
    with open(f"{OUT_DIR}/ALL_reports.txt", "w", encoding="utf-8") as f:       # one file for everything
        for name in DATASETS:
            p_ = f"{OUT_DIR}/{name}_report.txt"
            if os.path.exists(p_): f.write(open(p_, encoding="utf-8").read() + "\n\n" + "=" * 90 + "\n\n")
    print(summary_df.round(4).set_index("dataset").T)
    
    # ---------- zip the output folder for download ----------
    zip_path = f"{OUT_DIR}.zip"
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for fp in Path(OUT_DIR).rglob("*"):
            if fp.is_file():
                zf.write(fp, fp.relative_to(Path(OUT_DIR).parent))
    print(f"Zipped {OUT_DIR} -> {zip_path} ({os.path.getsize(zip_path) / 1024**2:.1f} MB)")


# Data Preparation

In [ ]:
if WILL_FT:
    # ---------- 1. definitions (from your prompt's intent_options) ----------
    
    ALL = sorted(DESC)
    assert len(ALL) == 60
    unknown = set(df1["intent"].dropna()) - set(ALL)
    assert not unknown, f"labels in df1 without a definition: {unknown}"
    df1 = df1.dropna(subset=["utt", "intent"]).copy()
    df1["intent_definition"] = df1["intent"].map(DESC)     # inspection column only
    
    dom = lambda l: l.split("_")[0]                        # matches the prompt's domain grouping
    by_dom = {}
    for l in ALL: by_dom.setdefault(dom(l), []).append(l)
    
    # ---------- 2. row builder ----------
    
    
    
    def make_gold(label):
        # TODO: print(dataset[0]["gold"]) on LocalLLaMA/typed-decisions and match that format exactly
        return json.dumps({"intent": label}, ensure_ascii=False)
    
    def sample_k(rng):
        r = rng.random()
        if r < 0.35: return 60                    # full label list
        if r < 0.75: return rng.randint(20, 40)   # medium
        return rng.randint(6, 20)                 # small
    
    def make_row(utt, gold, rng):
        k = sample_k(rng)
        sib = [l for l in by_dom[dom(gold)] if l != gold]
        chosen = {gold, *rng.sample(sib, min(len(sib), rng.randint(1, 3)))}   # same-domain hard negatives
        rest = [l for l in ALL if l not in chosen]
        chosen |= set(rng.sample(rest, max(0, min(k - len(chosen), len(rest)))))
        labels = list(chosen); rng.shuffle(labels)
        return {
            "state": str(utt),
            "questions": json.dumps({"intent": {
                "type": "choice", "instructions": INSTR,
                "criteria": {l: DESC[l] for l in labels},
            }}, ensure_ascii=False),
            "gold": make_gold(gold),
        }
    
    # ---- whole df1 -> training set ----
    df_all = df1.dropna(subset=["utt", "intent"]).reset_index(drop=True)
    print(len(df_all), "utterances;", df_all.intent.nunique(), "labels")
    print(df_all.intent.value_counts().tail(5))           # check no label is very rare
    
    rng = random.Random(1)
    COPIES = COPIES                                            # different option lists per utterance
    rows = [make_row(u, y, rng) for u, y in zip(df_all.utt, df_all.intent) for _ in range(COPIES)]
    train_ds = Dataset.from_list(rows)
    
    # ---- inspect ----
    print(train_ds[0]["state"])
    print(json.dumps(json.loads(train_ds[0]["questions"]), indent=2, ensure_ascii=False)[:1000])
    print(train_ds[0]["gold"])
    
    full = json.dumps({"intent": {"type": "choice", "instructions": INSTR, "criteria": DESC}}, ensure_ascii=False)
    # option 2: use the underlying text tokenizer (safer)
    tok = getattr(tokenizer, "tokenizer", tokenizer)
    print("full-60 options tokens:", len(tok(full)["input_ids"]), "| max_seq_length = 2048")
    # ---- build ----
    train_items, report = FastDecisionModel.build_dataset(train_ds, tokenizer, model)
    print(f"Skipped {report['skipped']} of {report['total']} decisions")   # should be ~0
    print(f"{len(train_items)} training rows")

# Train

<a name="Train"></a>
### Train the model
Now let's train our model. We do 60 steps (about 1.7 epochs) to speed things up, which reached 75% to 78% test accuracy in our runs and took about 2 hours on a free T4 GPU (12 minutes on an RTX PRO 6000). The T4 has no bfloat16 and Qwen3.5 gives NaNs in pure float16, so Unsloth trains it in float32 there, which is why the T4 is slow. For the full run, set `num_train_epochs = 2` and remove `max_steps`: that is 70 steps here, and it reached 78% test accuracy in 25 minutes on an A100.

In [ ]:
if WILL_FT:
    trainer = DecisionTrainer(
        model = model,
        processing_class = tokenizer,
        train_dataset = train_items,
        args = TrainingArguments(
            per_device_train_batch_size = 8,
            gradient_accumulation_steps = 4, # Use GA to mimic batch size!
            warmup_steps = 10,
            num_train_epochs = num_train_epochs, # Set this for 1 full training run.
            # max_steps = 20,
            learning_rate = 2e-4,
            lr_scheduler_type = "cosine",
            weight_decay = 0.01,
            bf16 = is_bfloat16_supported(),
            fp16 = not is_bfloat16_supported(),
            eval_strategy = "no",
            logging_steps = 10,
            output_dir = "outputs",
            report_to = "none", # Use TrackIO/WandB etc
            seed = 3407,
        ),
    )

In [ ]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

Let's train the model! To resume a training run, set `trainer.train(resume_from_checkpoint = True)`

In [ ]:
if WILL_FT:
    trainer_stats = trainer.train()

<a name="Inference"></a>
# Inference


In [ ]:
if WILL_FT:
    DATASETS = {                                   # name -> csv path (needs columns: utt, intent)
        "EN": EN_testing_dataset_with_intent_name,
        "BN": BN_testing_dataset_with_intent_name,
        "CM": CM_testing_dataset_with_intent_name,
        "Banglish": Banglish_testing_dataset_with_intent_name,
    }
    N_SAMPLES = {"EN": 2000, "BN": 2000, "CM": 2000, "Banglish":2000}   # None = use every row
    REMOVE_TRAIN_OVERLAP = True     # keep identical for base and fine-tuned runs so both score the same rows
    RUN_NAME = "finetuned"
    OUT_DIR = f"/kaggle/working/results_{RUN_NAME}_{LANG}"
    os.makedirs(OUT_DIR, exist_ok=True)
    DOM = lambda l: l.split("_")[0]
    # Needs from earlier cells: model, tokenizer, ALL, DESC, INSTR, df1, base_model_inference
    # ==================================================
    
    FastDecisionModel.for_inference(model)
    
    def predict_full(utt, labels=None):
        labels = labels or ALL
        questions = {"intent": {"type": "choice", "instructions": INSTR,
                                "criteria": {l: DESC[l] for l in labels}}}
        res = FastDecisionModel.predict(model, tokenizer, utt, questions)["intent"]
        answer = res.get("answer", res.get("label"))
        ranked = sorted(res["probabilities"].items(), key=lambda kv: -kv[1])
        return answer, ranked
    
    def ece_score(conf, correct, bins=10):
        edges = np.linspace(0, 1, bins + 1); e = 0.0
        for lo, hi in zip(edges[:-1], edges[1:]):
            m = (conf > lo) & (conf <= hi)
            if m.any(): e += m.mean() * abs(correct[m].mean() - conf[m].mean())
        return e
    
    def wilson(k, n, z=1.96):
        if n == 0: return 0.0, 0.0
        p = k / n; d = 1 + z * z / n
        c = (p + z * z / (2 * n)) / d
        h = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
        return c - h, c + h
    
    def load_partial(path):
        done = {}
        if os.path.exists(path):
            with open(path, encoding="utf-8") as f:
                for line in f:
                    try:
                        r = json.loads(line); done[(r["row_id"], r["utt"])] = r
                    except (json.JSONDecodeError, KeyError):
                        pass                                   # last line may be cut by a crash
        return done
    
    def run_dataset(name, path, n):
        df = pd.read_csv(path).dropna(subset=["utt", "intent"]).reset_index(drop=True)
        bad = set(df.intent) - set(ALL)
        assert not bad, f"{name}: labels without a definition: {bad}"
        removed = 0
        if REMOVE_TRAIN_OVERLAP:
            keep = ~df.utt.isin(set(df1.utt)); removed = int((~keep).sum()); df = df[keep].reset_index(drop=True)
        total_rows = len(df)
        sub = (df.sample(min(n, len(df)), random_state=0) if n else df).reset_index(drop=True)
    
        # ---------- predictions: appended line by line, resumable ----------
        ckpt = f"{OUT_DIR}/{name}_predictions_partial.jsonl"
        done = load_partial(ckpt)
        if done: print(f"{name}: resuming, {len(done)} predictions already saved")
        rows = list(done.values())
        correct_ct = sum(r["pred"] == r["intent"] for r in rows)
        todo = [(i, u, g) for i, (u, g) in enumerate(zip(sub.utt, sub.intent)) if (i, u) not in done]
    
        with open(ckpt, "a", encoding="utf-8") as f:
            bar = tqdm(todo, total=len(todo), desc=name, dynamic_ncols=True)
            for i, utt, gold in bar:
                t1 = time.time()
                pred, ranked = predict_full(utt)
                r = {"row_id": i, "utt": utt, "intent": gold, "pred": pred, "conf": ranked[0][1],
                     "top5": [l for l, _ in ranked[:5]], "sec": time.time() - t1}
                rows.append(r); correct_ct += int(pred == gold)
                bar.set_postfix(acc=f"{correct_ct / len(rows):.3f}")
                f.write(json.dumps(r, ensure_ascii=False) + "\n")
                f.flush(); os.fsync(f.fileno())
    
        res = pd.DataFrame(rows).sort_values("row_id").reset_index(drop=True)
        secs = res.sec.sum()
        y, p = res.intent.values, res.pred.values
        ok = (y == p); conf = res.conf.values.astype(float)
    
        # ---------- metrics ----------
        labs = sorted(set(y))
        P, R, F, _ = precision_recall_fscore_support(y, p, labels=labs, average=None, zero_division=0)
        macro = precision_recall_fscore_support(y, p, labels=labs, average="macro", zero_division=0)[:3]
        weighted = precision_recall_fscore_support(y, p, labels=labs, average="weighted", zero_division=0)[:3]
        acc = ok.mean(); lo, hi = wilson(int(ok.sum()), len(ok))
        top = {k: np.mean([g in t5[:k] for g, t5 in zip(y, res.top5)]) for k in (1, 3, 5)}
        domain_ok = np.array([DOM(a) == DOM(b) for a, b in zip(y, p)])
        invalid = int((~res.pred.isin(ALL)).sum())
        rng = np.random.default_rng(0); n_ = len(res); boot_acc, boot_f1 = [], []
        for _ in range(500):
            ix = rng.integers(0, n_, n_)
            boot_acc.append((y[ix] == p[ix]).mean())
            boot_f1.append(precision_recall_fscore_support(y[ix], p[ix], labels=labs, average="macro", zero_division=0)[2])
        err = res[~ok]
        intra = int(domain_ok[~ok].sum()); cross = int((~ok).sum()) - intra
        pairs = Counter(zip(err.intent, err.pred)).most_common(15)
        dom_acc = (res.assign(ok=ok, dom=[DOM(l) for l in y]).groupby("dom").ok
                   .agg(["mean", "count"]).sort_values("mean"))
        cls = pd.DataFrame({"label": labs, "precision": P, "recall": R, "f1": F,
                            "support": [int((y == l).sum()) for l in labs]}).sort_values("f1")
        cm = pd.crosstab(res.intent, res.pred)
    
        summary = {"run": RUN_NAME, "dataset": name, "n_eval": len(res), "accuracy": acc,
                   "acc_ci_low": lo, "acc_ci_high": hi, "top3_acc": top[3], "top5_acc": top[5],
                   "macro_P": macro[0], "macro_R": macro[1], "macro_F1": macro[2], "weighted_F1": weighted[2],
                   "kappa": cohen_kappa_score(y, p), "mcc": matthews_corrcoef(y, p),
                   "domain_acc": domain_ok.mean(), "ECE": ece_score(conf, ok.astype(float)),
                   "mean_conf_correct": conf[ok].mean() if ok.any() else np.nan,
                   "mean_conf_wrong": conf[~ok].mean() if (~ok).any() else np.nan,
                   "errors": int((~ok).sum()), "intra_domain_err": intra, "cross_domain_err": cross,
                   "invalid_preds": invalid, "sec_per_utt": secs / len(res)}
    
        # ---------- text report ----------
        L = [f"RUN: {RUN_NAME} | DATASET: {name} | file: {path}",
             f"rows after filtering: {total_rows} (train-overlap removed: {removed}) | evaluated: {len(res)}",
             f"inference time: {secs/60:.1f} min ({secs/len(res):.2f} s/utt) | label list: all {len(ALL)} labels per query", "",
             "=== OVERALL ===",
             f"Accuracy (= micro-F1)     : {acc:.4f}  (95% Wilson CI {lo:.4f}-{hi:.4f}; bootstrap {np.percentile(boot_acc,2.5):.4f}-{np.percentile(boot_acc,97.5):.4f})",
             f"Top-3 / Top-5 accuracy    : {top[3]:.4f} / {top[5]:.4f}",
             f"Macro    P / R / F1       : {macro[0]:.4f} / {macro[1]:.4f} / {macro[2]:.4f}  (F1 bootstrap CI {np.percentile(boot_f1,2.5):.4f}-{np.percentile(boot_f1,97.5):.4f})",
             f"Weighted P / R / F1       : {weighted[0]:.4f} / {weighted[1]:.4f} / {weighted[2]:.4f}",
             f"Cohen's kappa / MCC       : {summary['kappa']:.4f} / {summary['mcc']:.4f}",
             f"Domain-level accuracy     : {summary['domain_acc']:.4f}",
             f"Invalid (non-label) preds : {invalid}", "",
             "=== ERROR ANALYSIS ===",
             f"Total errors: {summary['errors']} | same-domain: {intra} | cross-domain: {cross}",
             f"Calibration: ECE={summary['ECE']:.4f} | mean confidence correct={summary['mean_conf_correct']:.3f}, wrong={summary['mean_conf_wrong']:.3f}", "",
             "Top confused pairs (gold -> pred : count):"]
        L += [f"  {a} -> {b} : {c}" for (a, b), c in pairs]
        L += ["", "Accuracy per domain (worst first):", dom_acc.round(4).to_string(), "",
              "10 weakest labels by F1:", cls.head(10).round(4).to_string(index=False), "",
              "10 strongest labels by F1:", cls.tail(10).round(4).to_string(index=False), "",
              "Sample errors (first 25):", err[["utt", "intent", "pred", "conf"]].head(25).round(3).to_string(index=False), "",
              "=== FULL PER-CLASS REPORT ===", classification_report(y, p, labels=labs, digits=4, zero_division=0)]
        txt = "\n".join(L)
    
        open(f"{OUT_DIR}/{name}_report.txt", "w", encoding="utf-8").write(txt)
        cls.to_csv(f"{OUT_DIR}/{name}_per_class.csv", index=False)
        cm.to_csv(f"{OUT_DIR}/{name}_confusion_matrix.csv")
        res.drop(columns=["top5", "sec"]).to_csv(f"{OUT_DIR}/{name}_predictions.csv", index=False, encoding="utf-8-sig")
        json.dump(summary, open(f"{OUT_DIR}/{name}_summary.json", "w"), default=float)   # marks dataset as finished
        print(txt[:2500])
        return summary
    
    # ---------- run all datasets (skips finished ones, resumes the interrupted one) ----------
    all_sum = []
    for name, path in DATASETS.items():
        sj = f"{OUT_DIR}/{name}_summary.json"
        if os.path.exists(sj):
            print(f"{name} already finished, skipping"); all_sum.append(json.load(open(sj))); continue
        all_sum.append(run_dataset(name, path, N_SAMPLES.get(name)))
        pd.DataFrame(all_sum).to_csv(f"{OUT_DIR}/summary_all_datasets.csv", index=False)
    
    summary_df = pd.DataFrame(all_sum)
    summary_df.to_csv(f"{OUT_DIR}/summary_all_datasets.csv", index=False)
    with open(f"{OUT_DIR}/ALL_reports.txt", "w", encoding="utf-8") as f:       # one file for everything
        for name in DATASETS:
            p_ = f"{OUT_DIR}/{name}_report.txt"
            if os.path.exists(p_): f.write(open(p_, encoding="utf-8").read() + "\n\n" + "=" * 90 + "\n\n")
    print(summary_df.round(4).set_index("dataset").T)

In [ ]:
# model.save_pretrained("qwen_lora")  # Local saving
if WILL_FT:
    model.push_to_hub(remote_repo, token = hf_token) # Online saving

In [ ]:
if False:
    from unsloth import FastDecisionModel
    model, tokenizer = FastDecisionModel.from_pretrained(
        model_name = "qwen_lora", # YOUR MODEL YOU USED FOR TRAINING
        max_seq_length = 2048,
        load_in_4bit = True,
    )